# 第 6 章　面板数据回归
## FE、HDFE、Fama-MacBeth 与 Portfolio Sort

::: {.callout-note}
## 本章要点

1. **固定效应（FE）的 FWL 理解**：组内去均值 = 对个体虚拟变量做 FWL 残差化
2. **FE vs RE 与 Hausman 检验**：什么时候 FE，什么时候 RE
3. **高维固定效应（HDFE）**：`reghdfe` + `pyfixest` 同时控制多维 FE
4. **Fama-MacBeth 回归**：每期截面回归 → 时序均值推断，资产定价检验标准
5. **Portfolio Sort 分组分析**：FM 的非参数对照工具
6. **Python + Stata 双语呈现**：`pyfixest` / `linearmodels` 对照 `reghdfe` + `xtfmb`

**与第 5 章的连接**：固定效应是 FWL 定理的直接应用——
加入个体虚拟变量并做 FWL 残差化，等价于组内去均值。
理解了这一点，HDFE 和 DID（第 8 章）就只是「换一种去均值方式」。
:::


## 环境准备


In [ ]:
# ── 第 6 章　面板数据回归　──────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, compare
import pyfixest as pf
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_CLEAN = 'data_clean'
OUTPUT     = 'output'
for d in [DATA_CLEAN, OUTPUT]:
    os.makedirs(d, exist_ok=True)

RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 1　本章演示数据

本章沿用第 5 章的借款利率面板，并扩充为更大的样本（500 家公司 × 8 年）
以便演示 HDFE 和 Fama-MacBeth。
同时植入一个**时不变的遗漏变量**（公司质量 `quality`），
用来演示「加入个体固定效应能消除时不变遗漏变量偏误」这一核心功能。


In [ ]:
# ── 1.1  读入真实数据 or 生成模拟面板 ───────────────────────────────────
try:
    df = pd.read_csv(f'{DATA_CLEAN}/merged_clean.csv', dtype={'stkcd': str})
    need = ['stkcd','year','soe','Size_w','Leverage_w','ROA_w','avg_loan_rate']
    df   = df.dropna(subset=[c for c in need if c in df.columns])
    DATA_SOURCE = 'CSMAR（真实数据）'
    print(f'真实数据：{df.shape}  来源：{DATA_SOURCE}')
except FileNotFoundError:
    n_firms, n_years = 500, 8
    n = n_firms * n_years
    firm_id  = np.repeat(np.arange(n_firms), n_years)
    year_id  = np.tile(np.arange(2016, 2016 + n_years), n_firms)

    # 时不变特征
    soe_firm     = RNG.binomial(1, 0.35, n_firms)
    quality_firm = RNG.normal(0, 1, n_firms)          # 遗漏变量
    size_base    = 1.5 * soe_firm + 0.6 * quality_firm + RNG.normal(0, 0.8, n_firms)

    soe       = np.repeat(soe_firm, n_years)
    quality   = np.repeat(quality_firm, n_years)
    Size_w    = np.repeat(size_base, n_years) + RNG.normal(0, 0.3, n)
    Leverage  = np.clip(
        np.repeat(RNG.beta(2,5,n_firms), n_years) - 0.08*soe + RNG.normal(0,0.05,n),
        0.05, 0.95)
    ROA_w     = RNG.normal(0.05, 0.03, n) - 0.02*soe + 0.03*quality

    # DGP：真实系数 soe=-0.5, size=-0.3, lev=+0.8, roa=-0.5, quality=-0.4
    eps  = RNG.normal(0, 0.3, n)
    rate = 5.0 - 0.5*soe - 0.3*Size_w + 0.8*Leverage - 0.5*ROA_w - 0.4*quality + eps

    # 时变行业冲击（用于演示年份 FE）
    year_shock = dict(zip(range(2016,2024),
                         RNG.normal(0, 0.2, n_years)))
    rate += np.array([year_shock[y] for y in year_id])

    df = pd.DataFrame({
        'stkcd': firm_id.astype(str).str.zfill(6),
        'year' : year_id,
        'soe'  : soe.astype(int),
        'quality': quality,          # 只在演示中可见，真实研究里观察不到
        'Size_w' : Size_w,
        'Leverage_w': Leverage,
        'ROA_w'  : ROA_w,
        'avg_loan_rate': rate,
    })
    DATA_SOURCE = '模拟数据（含时不变遗漏变量 quality）'
    print(f'模拟数据：{df.shape}')
    print(f'真实系数：soe=-0.5  size=-0.3  lev=+0.8  roa=-0.5  quality=-0.4')

print(f'\n样本：{df["stkcd"].nunique()} 家公司，{df["year"].nunique()} 年')


---

## 2　固定效应的 FWL 理解

### 2.1　时不变遗漏变量问题

在上面的 DGP 中，`quality`（公司质量）同时影响 `soe` 的取值
（质量好的公司更容易变成国企）和 `avg_loan_rate`（质量好利率低）。
但 `quality` 在真实研究中**观察不到**。

这意味着：即使我们控制了所有可观测的变量，
OLS 对 `soe` 系数的估计仍然有偏——
因为 `soe` 通过 `quality` 与扰动项相关。

**个体固定效应（Individual FE）**的核心功能：
控制所有**时不变**的个体特征，无论是否可观测。

### 2.2　FE = FWL 的特例

加入个体虚拟变量 $d_i$，回归变为：

$$Y_{it} = \alpha_i + X_{it}'\beta + \varepsilon_{it}$$

由 FWL 定理，$\hat{\beta}$ 等于先对所有变量做**组内去均值**再 OLS：

$$\tilde{Y}_{it} = Y_{it} - \bar{Y}_i, \quad
\tilde{X}_{it} = X_{it} - \bar{X}_i$$

$$\hat{\beta}_{FE} = (\tilde{X}'\tilde{X})^{-1}\tilde{X}'\tilde{Y}$$

::: {.callout-important}
## 固定效应「控制了什么」和「没控制什么」

**控制了**：所有时不变的个体特征（可观测的 + 不可观测的）
→ `quality`、创始人背景、地理位置……

**没控制**：随时间变化的遗漏变量
→ 如果「公司质量随时间改善且同时影响融资」，FE 仍然有偏

**代价**：时不变的解释变量（如 `soe`）的系数无法识别
→ 公司 FE 会吸收掉所有公司层面的时不变特征，包括 `soe`
:::


In [ ]:
# ── 2.3  演示：OLS vs FE 对遗漏变量偏误的处理 ──────────────────────────
controls = [c for c in ['Size_w','Leverage_w','ROA_w'] if c in df.columns]
ctrl_str = ' + '.join(controls)

# 模型一：OLS（忽略遗漏变量 quality）
m_ols = pf.feols(f'avg_loan_rate ~ soe + {ctrl_str}',
                  data=df, vcov={'CRV1':'stkcd'})

# 模型二：如果能观察到 quality（理想情况）
if 'quality' in df.columns:
    m_oracle = pf.feols(f'avg_loan_rate ~ soe + {ctrl_str} + quality',
                         data=df, vcov={'CRV1':'stkcd'})

# 模型三：公司 FE（消除所有时不变遗漏变量）
m_fe = pf.feols(f'avg_loan_rate ~ {ctrl_str} | stkcd',
                 data=df, vcov={'CRV1':'stkcd'})
# 注意：soe 是时不变变量，加入公司 FE 后无法识别（会被吸收）

# 模型四：双向 FE
m_twfe = pf.feols(f'avg_loan_rate ~ {ctrl_str} | stkcd + year',
                   data=df, vcov={'CRV1':'stkcd'})

print('OLS 对 soe 的估计（含遗漏变量偏误）：',
      round(m_ols.coef()['soe'], 4))
if 'quality' in df.columns:
    print('控制 quality 后（理想情况）     ：',
          round(m_oracle.coef()['soe'], 4))
print('真实系数（DGP）               ：-0.5000')
print()
print('公司 FE 模型：soe 被公司 FE 吸收，无法单独估计')
print('（这是 FE 的代价：时不变处理变量无法识别）')


In [ ]:
# ── 2.4  FWL 手动验证：组内去均值 = 加虚拟变量 ──────────────────────────
# 命题：公司 FE 回归的系数 = 手动组内去均值后 OLS 的系数

df_dem = df.copy()
for col in ['avg_loan_rate'] + controls:
    firm_mean = df_dem.groupby('stkcd')[col].transform('mean')
    df_dem[f'{col}_dem'] = df_dem[col] - firm_mean

ctrl_dem_str = ' + '.join([f'{c}_dem' for c in controls])
m_manual = smf.ols(f'avg_loan_rate_dem ~ {ctrl_dem_str} - 1', data=df_dem).fit()

print('FWL 验证（公司 FE）：')
for col in controls[:2]:  # 只对比前两个系数
    c_pyfixest = m_fe.coef().get(col, float('nan'))
    c_manual   = m_manual.params.get(f'{col}_dem', float('nan'))
    print(f'  {col:<14}: pyfixest={c_pyfixest:.6f}  手动去均值={c_manual:.6f}  ',
          f'差值={abs(c_pyfixest-c_manual):.2e}')
print('结论：两种方法完全一致，FE = FWL 特例得证 ✓')


---

## 3　FE vs RE 与 Hausman 检验

### 3.1　随机效应（RE）的假设

**随机效应（Random Effects）**假设个体效应 $\alpha_i$ 与解释变量**不相关**：

$$\text{Cov}(\alpha_i, X_{it}) = 0$$

如果这个假设成立，RE 比 FE 更有效率（标准误更小），
而且可以估计时不变变量（如 `soe`）的系数。

如果假设**不成立**（通常如此），RE 有偏，必须用 FE。

### 3.2　Hausman 检验

**Hausman 检验**的原假设：个体效应与解释变量不相关（RE 有效）。

- 若 $p > 0.05$：不能拒绝 $H_0$，可以用 RE
- 若 $p \leq 0.05$：拒绝 $H_0$，应该用 FE

::: {.callout-warning}
## Hausman 检验在实践中的局限

即使 Hausman 检验不显著，大多数经济金融研究仍然默认使用 FE，
理由是：面板数据里个体效应和解释变量相关几乎是必然的（比如公司质量和国企身份）。
RE 的使用需要额外的理论论证，而不仅仅是 Hausman 检验通过。
:::


In [ ]:
# ── 3.3  Python：FE vs RE 对比（linearmodels）────────────────────────────
# linearmodels 是 statsmodels 的面板数据扩展，语法与 R 的 plm 类似

# 设置面板索引（必须在 linearmodels 运行前完成）
df_panel = df.set_index(['stkcd', 'year'])

formula_lm = 'avg_loan_rate ~ soe + Size_w + Leverage_w + ROA_w + EntityEffects'

# 固定效应模型
fe_model = PanelOLS.from_formula(
    'avg_loan_rate ~ soe + Size_w + Leverage_w + ROA_w + EntityEffects',
    data=df_panel
).fit(cov_type='clustered', cluster_entity=True)

# 随机效应模型
re_model = RandomEffects.from_formula(
    'avg_loan_rate ~ soe + Size_w + Leverage_w + ROA_w',
    data=df_panel
).fit()

# 对比两个模型的系数
print('FE vs RE 系数对比：')
print(f'{'变量':<14} {'FE 系数':>10} {'RE 系数':>10} {'差值':>10}')
print('─' * 48)
for var in ['soe','Size_w','Leverage_w','ROA_w']:
    fe_c = fe_model.params.get(var, float('nan'))
    re_c = re_model.params.get(var, float('nan'))
    print(f'{var:<14} {fe_c:>10.4f} {re_c:>10.4f} {fe_c-re_c:>10.4f}')


In [ ]:
# ── 3.4  Hausman 检验 ────────────────────────────────────────────────────
# linearmodels 没有内置 Hausman，手动实现

def hausman_test(fe_result, re_result):
    '''Hausman 检验：H0 = 个体效应与解释变量不相关（RE 有效）'''
    vars_common = [v for v in fe_result.params.index
                   if v in re_result.params.index and v != 'Intercept']
    b_fe  = fe_result.params[vars_common].values
    b_re  = re_result.params[vars_common].values
    diff  = b_fe - b_re
    # 方差矩阵差
    V_fe = fe_result.cov.loc[vars_common, vars_common].values
    V_re = re_result.cov.loc[vars_common, vars_common].values
    V    = V_fe - V_re
    try:
        from numpy.linalg import pinv
        chi2 = float(diff @ pinv(V) @ diff)
        df_h = len(vars_common)
        p    = 1 - stats.chi2.cdf(chi2, df_h)
        return chi2, df_h, p
    except Exception:
        return None, None, None

chi2, df_h, p = hausman_test(fe_model, re_model)
if chi2 is not None:
    print(f'Hausman 检验：χ²({df_h}) = {chi2:.3f}  p = {p:.4f}')
    if p < 0.05:
        print('结论：拒绝 H0，应使用固定效应（FE）')
    else:
        print('结论：不能拒绝 H0，随机效应（RE）有效')
    if DATA_SOURCE.startswith('模拟'):
        print('（模拟数据已植入遗漏变量，预期应拒绝 H0）')


### 3.5　Stata 实现

```stata
%%stata
* ── FE vs RE 与 Hausman 检验 ─────────────────────────────────────────
use "data_temp/for_stata.dta", clear
xtset stkcd year      // 声明面板结构

* 固定效应
xtreg avg_loan_rate soe Size_w Leverage_w ROA_w, fe vce(cluster stkcd)
est store fe

* 随机效应
xtreg avg_loan_rate soe Size_w Leverage_w ROA_w, re
est store re

* Hausman 检验
hausman fe re
* 注：若提示矩阵非正定，加 sigmamore 选项：hausman fe re, sigmamore
```

**结果应与 Python 数值一致。**


---

## 4　高维固定效应（HDFE）

### 4.1　为什么需要 HDFE

有时候研究设计需要同时控制多个维度的固定效应，例如：
- 公司 FE + 年份 FE（双向固定效应，最常见）
- 公司 FE + 年份 FE + 行业-年份 FE
- 省份 FE + 行业 FE + 年份 FE

当固定效应的维度很高（比如 5000 家公司 × 8 年 × 申万行业），
直接加虚拟变量会产生数万个参数，计算量爆炸。

**HDFE 的解决方案**：利用 FWL 定理，用迭代去均值算法
（Alternating Projections / Gauss-Seidel）代替虚拟变量，
计算复杂度从 $O(n^3)$ 降至近似线性。

### 4.2　识别条件

多维 FE 的识别需要**连通性（connectedness）**：
不同固定效应组之间必须有足够的交叉观测。

例如：同时加公司 FE 和行业 FE，要求每家公司只属于一个行业——
这时行业 FE 完全被公司 FE 吸收，是**共线的**，必须删掉一个。

::: {.callout-warning}
## HDFE 的常见陷阱：过度控制导致解释变量被吸收

加入的 FE 维度越多，时不变或变化很慢的解释变量越容易被吸收。

经验原则：如果一个解释变量在某个 FE 维度内几乎不变
（比如 `soe` 在公司 FE 内完全不变），
该维度的 FE 会吸收掉该变量，系数会消失或方差极大。
:::


In [ ]:
# ── 4.3  Python：HDFE 实现（pyfixest）────────────────────────────────────
# pyfixest 使用 FWL + 迭代去均值，语法简洁

# 为演示添加行业变量（模拟数据中随机分配）
if 'industry' not in df.columns:
    n_ind = 10
    firm_to_ind = dict(zip(
        df['stkcd'].unique(),
        RNG.integers(0, n_ind, df['stkcd'].nunique())
    ))
    df['industry'] = df['stkcd'].map(firm_to_ind)

models = {
    '仅年份FE'        : 'avg_loan_rate ~ soe + Size_w + Leverage_w + ROA_w | year',
    '公司+年份（TWFE）' : 'avg_loan_rate ~ Size_w + Leverage_w + ROA_w | stkcd + year',
    '公司+行业×年份'   : 'avg_loan_rate ~ Size_w + Leverage_w + ROA_w | stkcd + industry^year',
}

fitted = {}
for label, formula in models.items():
    try:
        fitted[label] = pf.feols(formula, data=df, vcov={'CRV1':'stkcd'})
        n_obs = fitted[label].nobs
        print(f'{label:<20}: N={n_obs:,}  ✓')
    except Exception as e:
        print(f'{label:<20}: 失败 ({e})')

# 并排输出（只输出有 soe 的模型）
models_with_soe = [m for m in fitted.values() if 'soe' in m.coef().index]
if models_with_soe:
    pf.etable(models_with_soe, coef_fmt='b\n(se)', digits=3, stars=True)


In [ ]:
# ── 4.4  展示：不同 FE 设置对系数的影响 ─────────────────────────────────
# 用系数图展示 Size_w 在不同 FE 设置下的系数变化

coef_data = []
for label, m in fitted.items():
    if 'Size_w' in m.coef().index:
        c  = m.coef()['Size_w']
        se = m.se()['Size_w']
        coef_data.append({'模型': label, '系数': c,
                          'ci_lo': c - 1.96*se, 'ci_hi': c + 1.96*se})

if coef_data:
    cdf = pd.DataFrame(coef_data)
    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = range(len(cdf))
    ax.errorbar(cdf['系数'], y_pos,
               xerr=[cdf['系数']-cdf['ci_lo'], cdf['ci_hi']-cdf['系数']],
               fmt='o', color='steelblue', capsize=5, markersize=8)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    if DATA_SOURCE.startswith('模拟'):
        ax.axvline(-0.3, color='red', linewidth=1, linestyle=':',
                   label='真实系数 -0.3')
        ax.legend(fontsize=9)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(cdf['模型'], fontsize=9)
    ax.set_xlabel('Size_w 系数（点估计 + 95% CI）', fontsize=10)
    ax.set_title('不同固定效应设置下 Size_w 的系数', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch06_hdfe_coef.png', dpi=150, bbox_inches='tight')
    plt.show()


### 4.5　Stata 实现

```stata
%%stata
* ── HDFE：reghdfe（需要先安装：ssc install reghdfe, replace）──────────
use "data_temp/for_stata.dta", clear

* 双向固定效应（公司 + 年份）
reghdfe avg_loan_rate soe Size_w Leverage_w ROA_w, ///
    absorb(stkcd year) vce(cluster stkcd)
est store twfe

* 三维 FE：公司 + 行业×年份
reghdfe avg_loan_rate soe Size_w Leverage_w ROA_w, ///
    absorb(stkcd c.industry#c.year) vce(cluster stkcd)
est store hdfe3

esttab twfe hdfe3, star(* 0.1 ** 0.05 *** 0.01) se b(%9.4f) r2 scalars(N)
```

**`pyfixest` 与 `reghdfe` 结果应精确一致。**


---

## 5　Fama-MacBeth 回归

### 5.1　方法逻辑

**Fama-MacBeth（FM）回归**是资产定价实证研究的标准方法，
由 Fama & MacBeth（1973）提出，用于检验截面收益率是否由某些特征解释。

**两步法：**

1. **第一步：截面回归**
   在每一期 $t$，对截面数据做回归：
   $$R_{it} = \lambda_{0t} + \lambda_{1t} \text{Size}_{it-1} + \cdots + \varepsilon_{it}$$
   得到每期的系数 $\hat{\lambda}_{1t}, \hat{\lambda}_{2t}, \ldots$

2. **第二步：时序推断**
   对各期系数取均值，用时序标准差构造 $t$ 统计量：
   $$\bar{\lambda}_1 = \frac{1}{T}\sum_t \hat{\lambda}_{1t}, \quad
   t = \frac{\bar{\lambda}_1}{\text{SD}(\hat{\lambda}_{1t}) / \sqrt{T}}$$

### 5.2　FM vs 面板 FE 的比较

| 维度 | Fama-MacBeth | 面板 FE |
|------|-------------|---------|
| 目的 | 资产定价因子检验 | 因果效应估计 |
| 标准误 | 基于系数的时序波动 | 聚类标准误 |
| 个体 FE | 不控制 | 控制 |
| 时序相关 | 通过时序均值处理 | 需要 AC-robust SE |
| 样本量要求 | 需要足够多期（$T$ 大）| 需要足够多个体（$N$ 大）|


In [ ]:
# ── 5.3  Python：Fama-MacBeth 实现 ──────────────────────────────────────

def fama_macbeth(df, y, xs, entity='stkcd', time='year'):
    '''
    Fama-MacBeth 回归。
    返回：各期系数均值、Newey-West 调整后的 t 值和 p 值。
    '''
    coefs_by_t = []
    for t, gdf in df.groupby(time):
        gdf = gdf[[y] + xs].dropna()
        if len(gdf) < len(xs) + 2:
            continue
        m = smf.ols(f'{y} ~ ' + ' + '.join(xs), data=gdf).fit()
        coefs_by_t.append(m.params)
    coef_df = pd.DataFrame(coefs_by_t)

    T = len(coef_df)
    results = []
    for var in coef_df.columns:
        lam   = coef_df[var]
        mean  = lam.mean()
        # Newey-West 标准误（lag=1）
        nw_se = sm.OLS(lam, np.ones(T)).fit(
            cov_type='HAC', cov_kwds={'maxlags': 1}).bse[0]
        t_val = mean / nw_se
        p_val = 2 * (1 - stats.t.cdf(abs(t_val), df=T-1))
        results.append({'变量': var, '均值λ': round(mean,4),
                        'NW_SE': round(nw_se,4),
                        't值': round(t_val,3),
                        'p值': round(p_val,4)})
    return pd.DataFrame(results).set_index('变量')

xs_fm = [c for c in ['soe','Size_w','Leverage_w','ROA_w'] if c in df.columns]
fm_result = fama_macbeth(df, 'avg_loan_rate', xs_fm)
print('Fama-MacBeth 回归结果（Newey-West 调整标准误）：')
print(fm_result.to_string())


In [ ]:
# ── 5.4  用 linearmodels 的 FamaMacBeth 接口（更简洁）──────────────────
from linearmodels.panel import FamaMacBeth as FM

df_panel = df.set_index(['stkcd','year'])
fm_lm = FM.from_formula(
    'avg_loan_rate ~ soe + Size_w + Leverage_w + ROA_w',
    data=df_panel
).fit(bandwidth=1)   # bandwidth=1 对应 Newey-West lag=1

print(fm_lm.summary.tables[1])


### 5.5　Stata 实现

```stata
%%stata
* ── Fama-MacBeth 回归 ───────────────────────────────────────────────────
* 需要安装：ssc install xtfmb, replace
use "data_temp/for_stata.dta", clear
xtset stkcd year

* 方法一：xtfmb（推荐，自带 Newey-West 调整）
xtfmb avg_loan_rate soe Size_w Leverage_w ROA_w, lag(1)

* 方法二：手动逐期截面回归（等价）
* forvalues y = 2016/2023 {
*     reg avg_loan_rate soe Size_w Leverage_w ROA_w if year == `y'
*     est store m`y'
* }
```

**结果应与 `linearmodels.FamaMacBeth` 的均值系数一致。**


---

## 6　Portfolio Sort 分组分析

### 6.1　什么是 Portfolio Sort

Portfolio Sort 是 FM 回归的**非参数替代**，
通过直接比较极端分组的收益差来检验因子效应，
不需要假设关系是线性的。

**标准流程（以检验「规模效应」为例）：**

1. 每期按 `Size_w` 从小到大排序，分成 5 组（Quintile）或 10 组（Decile）
2. 计算每组下一期的平均利率（或超额收益）
3. 比较第 5 组（最大规模）和第 1 组（最小规模）的均值差（**5-1 spread**）
4. 对 5-1 spread 做时序 $t$ 检验

### 6.2　与 FM 的关系

- FM 假设特征与收益的关系是**线性**的，Portfolio Sort 不假设
- FM 可以同时控制多个特征，Portfolio Sort 做多变量控制需要**双重分组**（Double Sort）
- 两者结论一致时，增加说服力；不一致时，说明关系可能是非线性的


In [ ]:
# ── 6.3  单变量 Portfolio Sort ───────────────────────────────────────────

def portfolio_sort(df, sort_var, ret_var, n_groups=5,
                   entity='stkcd', time='year'):
    '''
    单变量分组分析。
    每期按 sort_var 分成 n_groups 组，
    返回各组的平均 ret_var 和 5-1（或 N-1）spread 的 t 检验结果。
    '''
    df = df.copy()
    spreads = []
    group_rets = {g: [] for g in range(1, n_groups+1)}

    for t, gdf in df.groupby(time):
        gdf = gdf[[sort_var, ret_var]].dropna()
        if len(gdf) < n_groups * 2:
            continue
        gdf['group'] = pd.qcut(gdf[sort_var], n_groups,
                                labels=range(1, n_groups+1))
        means = gdf.groupby('group')[ret_var].mean()
        for g in range(1, n_groups+1):
            if g in means.index:
                group_rets[g].append(means[g])
        if n_groups in means.index and 1 in means.index:
            spreads.append(means[n_groups] - means[1])

    # 各组均值
    group_means = {g: np.mean(v) for g, v in group_rets.items() if v}

    # Spread 的 t 检验
    if len(spreads) >= 3:
        t_stat, p_val = stats.ttest_1samp(spreads, 0)
    else:
        t_stat, p_val = float('nan'), float('nan')

    return group_means, np.mean(spreads) if spreads else float('nan'), t_stat, p_val

# 按规模分组，分析各组平均借款利率
if 'Size_w' in df.columns:
    group_means, spread, t, p = portfolio_sort(df, 'Size_w', 'avg_loan_rate')
    print('规模分组 → 平均借款利率：')
    print(f'  {'分组':<6} {'平均利率（%）':>12}')
    print('  ' + '─' * 22)
    for g, m in sorted(group_means.items()):
        label = '（最小）' if g==1 else '（最大）' if g==max(group_means) else ''
        print(f'  Q{g}{label:<6} {m:>12.4f}')
    print(f'  {'5-1 Spread':<10} {spread:>10.4f}%  t={t:.3f}  p={p:.4f}')
    sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
    print(f'  显著性：{sig if sig else "不显著"}')


In [ ]:
# ── 6.4  可视化：分组均值图 ─────────────────────────────────────────────
if 'Size_w' in df.columns:
    gm_df = pd.Series(group_means).reset_index()
    gm_df.columns = ['分组', '平均利率']

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(gm_df['分组'].astype(str).map(
                      lambda x: f'Q{x}'),
                  gm_df['平均利率'],
                  color='steelblue', edgecolor='white', width=0.6)
    ax.axhline(df['avg_loan_rate'].mean(), color='red', linestyle='--',
               linewidth=1, label='全样本均值')
    ax.set_xlabel('规模分组（Q1=最小，Q5=最大）', fontsize=10)
    ax.set_ylabel('平均借款利率（%）', fontsize=10)
    ax.set_title('Portfolio Sort：规模分组与借款利率', fontsize=11)
    ax.legend(fontsize=9)
    # 标注 spread
    ax.annotate(f'5-1 Spread = {spread:.3f}%\n(t={t:.2f}, p={p:.3f})',
               xy=(0.97, 0.05), xycoords='axes fraction',
               ha='right', va='bottom', fontsize=9,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch06_portfolio_sort.png', dpi=150, bbox_inches='tight')
    plt.show()


::: {.callout-tip}
## 提示词模板：Portfolio Sort + FM 完整资产定价检验

这是资产定价实证研究最常用的组合，整个 T4 作业都可以套用：

````
我有一个面板 DataFrame df，列为：
entity（公司代码）、time（年份）、
ret（下一期收益率）、以及若干特征变量（Size, BM, Momentum, ...）。

请帮我完成完整的资产定价截面检验：

第一步：Portfolio Sort
- 按 Size 每期分成 5 组（Quintile）
- 计算各组下期平均收益率
- 对 Q5-Q1 spread 做时序 t 检验（Newey-West 调整，lag=1）
- 绘制分组均值柱状图

第二步：Fama-MacBeth 回归
- 每期截面回归：ret ~ Size + BM + Momentum
- 对各期系数取均值，用 Newey-West 标准误做 t 检验
- 输出系数均值、t 值、p 值

第三步：双重分组（Double Sort，控制 BM 后检验 Size 效应）
- 先按 BM 分成 3 组，在每个 BM 组内再按 Size 分成 3 组
- 输出 3×3 的均值矩阵和行/列的 spread
````
:::


---

## 7　综合案例：借款利率影响因素的完整面板分析


In [ ]:
# ── 7.1  完整分析：从 OLS 到 TWFE ───────────────────────────────────────
print('=' * 65)
print(f'数据：{DATA_SOURCE}')
print(f'样本：{df["stkcd"].nunique()} 家公司，{df["year"].nunique()} 年，N={len(df)}')
print('=' * 65)

base_ctrl = [c for c in ['Size_w','Leverage_w','ROA_w'] if c in df.columns]
ctrl_f    = ' + '.join(base_ctrl)

# 四个递进模型
reg1 = pf.feols(f'avg_loan_rate ~ soe + {ctrl_f}',
                 data=df, vcov={'CRV1':'stkcd'})
reg2 = pf.feols(f'avg_loan_rate ~ soe + {ctrl_f} | year',
                 data=df, vcov={'CRV1':'stkcd'})
# 注：加入公司 FE 后 soe 被吸收（时不变），改用 Leverage 等时变变量展示
reg3 = pf.feols(f'avg_loan_rate ~ {ctrl_f} | stkcd',
                 data=df, vcov={'CRV1':'stkcd'})
reg4 = pf.feols(f'avg_loan_rate ~ {ctrl_f} | stkcd + year',
                 data=df, vcov={'CRV1':'stkcd'})

pf.etable(
    [reg1, reg2, reg3, reg4],
    coef_fmt='b\n(se)', digits=3, stars=True,
    labels={'avg_loan_rate':'借款利率（%）','soe':'国企（=1）',
            'Size_w':'规模','Leverage_w':'杠杆率','ROA_w':'ROA'}
)


In [ ]:
# ── 7.2  规律总结 ────────────────────────────────────────────────────────
lev_ols   = reg1.coef()['Leverage_w']
lev_twfe  = reg4.coef()['Leverage_w']
pct_diff  = (lev_twfe - lev_ols) / abs(lev_ols) * 100

print('\n── 关键系数变化（以 Leverage_w 为例）────────────────────────')
print(f'OLS（无 FE）              ：{lev_ols:.4f}')
print(f'TWFE（公司+年份 FE）      ：{lev_twfe:.4f}')
print(f'变化幅度                  ：{pct_diff:+.1f}%')
if DATA_SOURCE.startswith('模拟'):
    print(f'真实系数（DGP）          ：+0.8000')
    print(f'TWFE 估计误差           ：{lev_twfe - 0.8:.4f}（应接近 0）')
print()
print('解读：')
print('OLS 混入了公司固有特征（quality）和宏观年份效应的影响。')
print('TWFE 通过去均值消除了这两类影响，得到更干净的偏效应估计。')
print('但 TWFE 无法估计时不变变量（soe）的系数——这是它的代价。')
print('要估计 soe 的因果效应，需要 DID 或 IV（见第 8 章）。')


---

## 8　章末练习

**练习 1（FE = FWL 验证）**
对双向 FE（公司 + 年份）模型，手动执行二维去均值：
先公司内去均值，再年份内去均值，循环迭代至收敛。
验证手动去均值后的 OLS 系数与 `pyfixest` TWFE 系数一致。
写一句话解释：为什么单次双向去均值不精确，需要迭代？

**练习 2（Hausman 检验解读）**
对真实数据（或本章模拟数据）做 Hausman 检验。
结合理论：在「上市公司借款利率」这个研究场景中，
你认为 FE 还是 RE 更合适？Hausman 检验结果是否支持你的判断？

**练习 3（FM + Portfolio Sort）**
用 `ROA_w` 作为排序变量，完成：
① 单变量 Portfolio Sort（5 组），绘制分组均值图，报告 5-1 spread 的 $t$ 值
② Fama-MacBeth 回归（结果变量：`avg_loan_rate`，解释变量：`ROA_w` + 其他特征）
③ 比较两种方法的结论是否一致，并说明为什么可能不同

**练习 4（思考题）**
加入公司固定效应后，`soe` 的系数消失了（被吸收）。
这是否意味着「国企效应不存在」？
如果你坚持认为国企身份有因果效应，
在面板数据框架下，什么情形下可以识别时不变处理变量的效应？
（提示：想想 DID——如果某些公司在样本期内发生了国企→民企的转变……）
